In [1]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Define the image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Resize images to 224x224
    transforms.ToTensor(),           # Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize with ImageNet mean and std
])

# Custom dataset class
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (string): Directory with all the images, organized in subfolders per class.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {}

        # Scan the root directory to collect image paths and their respective labels
        for idx, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):  # Only process directories
                self.class_to_idx[class_name] = idx
                for img_name in os.listdir(class_path):
                    img_path = os.path.join(class_path, img_name)
                    if img_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                        self.image_paths.append(img_path)
                        self.labels.append(idx)
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")  # Ensure image is in RGB format
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        
        return image, label

# Set your data directory here
root_dir = r"E:\1 Paper Work\Cutting Tool Paper\Dataset\cutting tool data\test_data_40_images"
dataset = CustomImageDataset(root_dir=root_dir, transform=transform)

# Checking the number of samples and class labels
print("Number of samples:", len(dataset))
print("Class to index mapping:", dataset.class_to_idx)


Number of samples: 280
Class to index mapping: {'BF': 0, 'BFI': 1, 'GF': 2, 'GFI': 3, 'N': 4, 'NI': 5, 'TF': 6}


In [ ]:
from torch.utils.data import random_split

# Define the number of support samples per class
support_samples_per_class = 5

# Separate the dataset by class for splitting
class_indices = {i: [] for i in range(len(dataset.class_to_idx))}
for idx, (_, label) in enumerate(dataset):
    class_indices[label].append(idx)

# Create support and query indices
support_indices = []
query_indices = []

for class_id, indices in class_indices.items():
    support_indices.extend(indices[:support_samples_per_class])
    query_indices.extend(indices[support_samples_per_class:])

# Subset the dataset to create support and query sets
support_set = torch.utils.data.Subset(dataset, support_indices)
query_set = torch.utils.data.Subset(dataset, query_indices)

# Create DataLoaders for both sets
batch_size = 8  # You can adjust this
support_loader = DataLoader(support_set, batch_size=batch_size, shuffle=True)
query_loader = DataLoader(query_set, batch_size=batch_size, shuffle=False)

# Checking the sizes of each set
print("Support set size:", len(support_loader.dataset))
print("Query set size:", len(query_loader.dataset))


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Define a simple CNN backbone for feature extraction
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        # Load a pre-trained ResNet18 model
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # Remove the final fully connected layer to get feature embeddings instead of class probabilities
        self.backbone.fc = nn.Identity()
    
    def forward(self, x):
        # Forward pass through the backbone to get feature embeddings
        return self.backbone(x)

# Instantiate the feature extractor model
feature_extractor = FeatureExtractor()

# Test a dummy forward pass
sample_input = torch.randn(1, 3, 224, 224)  # A dummy image with batch size 1
sample_output = feature_extractor(sample_input)

print("Feature embedding size:", sample_output.shape)


In [ ]:
import torch

def compute_prototypes(features, labels, n_classes):
    """
    Compute the prototypes (mean feature vector) for each class in the support set.
    
    Args:
        features (Tensor): Feature embeddings of the support images (shape: [num_support, 512]).
        labels (Tensor): Labels for the support images (shape: [num_support]).
        n_classes (int): Number of classes.
    
    Returns:
        Tensor: Class prototypes (shape: [n_classes, 512]).
    """
    prototypes = []
    for i in range(n_classes):
        class_features = features[labels == i]  # Select features of the current class
        prototype = class_features.mean(0)      # Calculate the mean to get the prototype
        prototypes.append(prototype)
    return torch.stack(prototypes)

def few_shot_classification(model, support_loader, query_loader, n_classes):
    """
    Classify query images based on distances to class prototypes.

    Args:
        model (nn.Module): The feature extractor model.
        support_loader (DataLoader): DataLoader for the support set.
        query_loader (DataLoader): DataLoader for the query set.
        n_classes (int): Number of classes.

    Returns:
        List[int], List[int]: True labels and predicted labels for the query set.
    """
    model.eval()
    with torch.no_grad():
        # Step 1: Extract features and labels for support images
        support_features = []
        support_labels = []
        for images, labels in support_loader:
            features = model(images)
            support_features.append(features)
            support_labels.append(labels)
        
        support_features = torch.cat(support_features)
        support_labels = torch.cat(support_labels)

        # Step 2: Compute class prototypes
        prototypes = compute_prototypes(support_features, support_labels, n_classes)

        # Step 3: Classify query images
        all_true_labels = []
        all_predicted_labels = []
        
        for images, labels in query_loader:
            query_features = model(images)
            dists = torch.cdist(query_features, prototypes)
            predicted_labels = dists.argmin(dim=1)
            all_true_labels.extend(labels.cpu().numpy())
            all_predicted_labels.extend(predicted_labels.cpu().numpy())
    
    return all_true_labels, all_predicted_labels

# Run few-shot classification using support and query loaders
n_classes = 4  # Number of classes: BF, GF, N, TF
true_labels, predicted_labels = few_shot_classification(feature_extractor, support_loader, query_loader, n_classes)

# Check output
print("True labels:", true_labels)
print("Predicted labels:", predicted_labels)


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Define class names for better readability in the matrix
class_names = ['BF', 'GF', 'N', 'TF']

# Generate the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Convert true labels to a tensor for plotting
true_labels_tensor = torch.tensor(true_labels)

# Extract features from the query set using the feature extractor
query_features = []
for images, _ in query_loader:
    with torch.no_grad():
        features = feature_extractor(images)
        query_features.append(features)

# Concatenate all query features
query_features = torch.cat(query_features).cpu().numpy()

# Perform t-SNE to reduce to 2D for visualization
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(query_features)

# Plot the t-SNE result
plt.figure(figsize=(6, 4))
for class_id in range(n_classes):
    indices = true_labels_tensor == class_id
    plt.scatter(tsne_results[indices, 0], tsne_results[indices, 1], label=class_names[class_id], alpha=0.6)
plt.legend()
plt.title('t-SNE Visualization of Feature Embeddings')
plt.show()


In [ ]:
import torch.optim as optim

# Unfreeze the last few layers of ResNet18 for fine-tuning
for name, param in feature_extractor.backbone.named_parameters():
    if "layer4" in name or "fc" in name:  # Only unfreeze layer4 and the final fully connected layer
        param.requires_grad = True
    else:
        param.requires_grad = False

# Add a new classification layer for training
class FineTunedModel(nn.Module):
    def __init__(self, feature_extractor, n_classes):
        super(FineTunedModel, self).__init__()
        self.feature_extractor = feature_extractor
        self.fc = nn.Linear(512, n_classes)  # Map 512-dimensional features to n_classes output
        
    def forward(self, x):
        features = self.feature_extractor(x)
        return self.fc(features)

# Instantiate the model
n_classes = 4  # BF, GF, N, TF
finetuned_model = FineTunedModel(feature_extractor, n_classes)

# Set up the optimizer and loss function
optimizer = optim.Adam(finetuned_model.parameters(), lr=1e-4)  # Adjust learning rate if needed
criterion = nn.CrossEntropyLoss()


In [ ]:
# Training loop for fine-tuning
num_epochs = 10  # Adjust as needed
finetuned_model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in support_loader:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = finetuned_model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(support_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("Fine-tuning complete.")


In [ ]:
# Re-evaluate the model using the few-shot classification and generate updated prototypes
true_labels, predicted_labels = few_shot_classification(finetuned_model.feature_extractor, support_loader, query_loader, n_classes)

# Confusion Matrix
plot_confusion_matrix(true_labels, predicted_labels, class_names=['BF', 'GF', 'N', 'TF'])

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def plot_tsne(features, labels, class_names):
    """
    Plots a 2D t-SNE visualization of the feature embeddings.
    
    Args:
        features (numpy.ndarray): Feature embeddings as a NumPy array (shape: [num_samples, feature_dim]).
        labels (torch.Tensor): True labels as a tensor (shape: [num_samples]).
        class_names (list of str): List of class names for labeling the plot.
    """
    tsne = TSNE(n_components=2, random_state=42)
    tsne_result = tsne.fit_transform(features)  # Directly use features, which is already a NumPy array

    plt.figure(figsize=(5, 4))
    for class_id in range(len(class_names)):
        indices = (labels == class_id).numpy()  # Convert labels to NumPy array for indexing
        plt.scatter(tsne_result[indices, 0], tsne_result[indices, 1], label=class_names[class_id], alpha=0.6)
    plt.legend()
    plt.title('t-SNE Visualization of Feature Embeddings')
    plt.show()


In [ ]:
# t-SNE Visualization after fine-tuning
query_features = []
for images, _ in query_loader:
    with torch.no_grad():
        features = finetuned_model.feature_extractor(images)
        query_features.append(features)

query_features = torch.cat(query_features).numpy()  # Already a NumPy array
plot_tsne(query_features, torch.tensor(true_labels), class_names=['BF', 'GF', 'N', 'TF'])


In [ ]:
import matplotlib.pyplot as plt

# Reinitialize the model and optimizer for a fresh fine-tuning
finetuned_model = FineTunedModel(feature_extractor, n_classes)
optimizer = optim.Adam(finetuned_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Lists to store losses
train_losses = []

# Training loop for fine-tuning with loss tracking
num_epochs = 10
finetuned_model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in support_loader:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = finetuned_model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(support_loader)
    train_losses.append(avg_loss)  # Store the training loss
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("Fine-tuning complete.")


In [ ]:
# Plotting the learning curve
plt.figure(figsize=(6, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Learning Curve')
plt.legend()
plt.show()


In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

# Reinitialize the model and optimizer for a fresh fine-tuning
finetuned_model = FineTunedModel(feature_extractor, n_classes)
optimizer = optim.Adam(finetuned_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Lists to store training and validation losses
train_losses = []
val_losses = []

num_epochs = 10
finetuned_model.train()
for epoch in range(num_epochs):
    running_train_loss = 0.0
    running_val_loss = 0.0

    # Training phase
    finetuned_model.train()
    for images, labels in support_loader:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = finetuned_model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()
    
    avg_train_loss = running_train_loss / len(support_loader)
    train_losses.append(avg_train_loss)  # Store the training loss

    # Validation phase
    finetuned_model.eval()
    with torch.no_grad():
        for images, labels in query_loader:
            outputs = finetuned_model(images)
            val_loss = criterion(outputs, labels)
            running_val_loss += val_loss.item()
    
    avg_val_loss = running_val_loss / len(query_loader)
    val_losses.append(avg_val_loss)  # Store the validation loss

    print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

print("Fine-tuning with validation tracking complete.")


In [ ]:
# Plotting the training and validation loss curves
plt.figure(figsize=(6, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss', marker='o')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curve')
plt.legend()
#plt.grid(True)
plt.show()


In [ ]:
import torch.nn.functional as F

# Get predicted probabilities for the query set
predicted_probs = []
true_labels = []
for images, labels in query_loader:
    with torch.no_grad():
        outputs = finetuned_model(images)
        probs = F.softmax(outputs, dim=1)  # Convert logits to probabilities
        predicted_probs.append(probs)
        true_labels.extend(labels.cpu().numpy())

predicted_probs = torch.cat(predicted_probs).cpu().numpy()


In [ ]:
from sklearn.metrics import roc_curve
import numpy as np
import matplotlib.pyplot as plt

def plot_cumulative_gain(true_labels, predicted_probs, class_names):
    n_classes = len(class_names)
    plt.figure(figsize=(8, 5))

    for i in range(n_classes):
        # Binarize the output for the current class
        binary_true_labels = (np.array(true_labels) == i).astype(int)
        
        # Sort by predicted probability for the class
        sorted_indices = np.argsort(predicted_probs[:, i])[::-1]
        sorted_true_labels = binary_true_labels[sorted_indices]

        # Cumulative gain
        cumulative_gain = np.cumsum(sorted_true_labels) / np.sum(sorted_true_labels)
        baseline = np.linspace(0, 1, len(cumulative_gain))

        # Plot cumulative gain
        plt.subplot(1, 2, 1)
        plt.plot(cumulative_gain, label=f'Cumulative Gain - {class_names[i]}')
        
        # Plot lift
        plt.subplot(1, 2, 2)
        lift = cumulative_gain / baseline
        plt.plot(lift, label=f'Lift - {class_names[i]}')

    # Final adjustments for plots
    plt.subplot(1, 2, 1)
    plt.plot(baseline, '--', color='gray', label='Baseline')
    plt.title('Cumulative Gain Chart')
    plt.xlabel('Percentage of Sample')
    plt.ylabel('Cumulative Gain')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(baseline, '--', color='gray', label='Baseline')
    plt.title('Lift Chart')
    plt.xlabel('Percentage of Sample')
    plt.ylabel('Lift')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Run the plotting function
plot_cumulative_gain(true_labels, predicted_probs, class_names=['BF', 'GF', 'N', 'TF'])
